# 08 · RAG: the document corpus, chunking and search

**Who this is for:** you, before building the assistant (Phase 4). No model training needed; notebooks 00-07 aren't required.
**You should already know:** basic Python and pandas.

**By the end you will be able to:**
- explain what **RAG** (retrieval-augmented generation) is, and when *not* to use it
- **chunk** documents so each piece is meaningful and citable
- explain **embeddings** and **cosine similarity**, and build a **FAISS** index
- compare **semantic**, **keyword (BM25)** and **hybrid** search
- **measure** retrieval with an eval set (hit@k, MRR) instead of eyeballing it
- handle **document versions** and decide when the assistant should **decline** to answer
- assemble the **prompt with citations** that the LLM will receive

**Outline**
1. Setup
2. What RAG is (and isn't) for
3. The corpus
4. Chunking: naive vs structure-aware
5. Embeddings in five minutes
6. A FAISS index and the first searches
7. Keyword and hybrid search
8. Measuring retrieval
9. Document versions
10. Knowing when not to answer
11. The prompt the LLM sees
12. Exercise
13. Pitfalls and next steps

Code used: `heavden/agent/corpus.py`, `heavden/agent/retrieval.py`, eval set `heavden/agent/evalsets/retrieval.yaml` (tests in `tests/test_rag.py`). Documents: `docs/corpus/`.

The embedding model (BAAI/bge-small-en-v1.5, ~130 MB) downloads once into `data/models/`. It needs the `rag` dependency group (`uv sync --all-groups`).

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from heavden.agent import corpus, retrieval

pd.set_option("display.max_colwidth", 90)


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
CORPUS = ROOT / "docs" / "corpus"
EVALSET = ROOT / "databricks" / "src" / "heavden" / "agent" / "evalsets" / "retrieval.yaml"

## 2. What RAG is (and isn't) for

An LLM knows nothing about the HeavDen hospitals: their escalation protocol, their devices, their model. **RAG** fixes that without retraining the LLM:

```
question ──► retrieve the most relevant passages ──► LLM answers from those passages, citing them
```

It works well for **documents**: protocols, manuals, runbooks. It works badly for **numbers in tables**: "how many high-risk patients are at Site B?" needs a SQL query, not a passage that happens to mention Site B. That's why the assistant has three tools (Plan.md §11): Genie for SQL, RAG for documents, and `explain_patient_risk` for the model. The original plan embedded table rows; the review in Plan.md §2 replaced that.

The quality of a RAG answer is capped by the quality of **retrieval**: if the right passage isn't found, the LLM can't use it. This notebook is about retrieval.

## 3. The corpus

Nine fictional documents, each with front matter (`doc_id`, `version`, `status`, `effective_date`). Their facts match the simulation (sites, devices, drift scenarios, model results), so the assistant's answers will agree with the data.

In [ ]:
docs = corpus.load_corpus(CORPUS)
pd.DataFrame(
    [
        {
            "doc_id": d.doc_id,
            "title": d.title,
            "version": d.version,
            "status": d.status,
            "effective": d.effective_date,
            "words": len(d.body.split()),
            "file": Path(d.path).name,
        }
        for d in docs
    ]
).sort_values(["doc_id", "version"])

Note the **two versions** of the escalation protocol (HD-CLIN-001): v1.0 is superseded by v2.0 on 2026-11-09, the day of the concept-drift scenario. Section 9 shows why that matters.

## 4. Chunking: naive vs structure-aware

Whole documents are too long to embed well and too long to cite precisely, so we split them into **chunks**.

**Naive chunking** cuts every N words, ignoring structure:

In [ ]:
news2 = next(d for d in docs if d.doc_id == "HD-CLIN-002")
naive = corpus.chunk_fixed(news2, size_words=120, overlap_words=20)
print(f"{len(naive)} naive chunks; chunk 2 starts and ends like this:\n")
words = naive[2].text.split()
print("START:", " ".join(words[:25]), "...")
print("END:  ...", " ".join(words[-25:]))

The cut lands mid-table and mid-sentence. A question about scoring could retrieve half of the scoring table.

**Structure-aware chunking** (`corpus.chunk_markdown`) follows the document:
- a chunk never crosses a heading, so it has one topic;
- tables and code blocks are never split;
- tiny pieces are merged and long sections are split at paragraph boundaries (at most ~220 words);
- each chunk keeps its **heading path**, which becomes its **citation**;
- the boilerplate "fictional document" banner is dropped (it would match every query).

In [ ]:
chunks = corpus.chunk_corpus(docs)
sizes = pd.Series([len(c.text.split()) for c in chunks])
print(
    f"{len(chunks)} chunks from {len(docs)} documents; words per chunk: "
    f"median {sizes.median():.0f}, max {sizes.max()}"
)

example = next(c for c in chunks if c.doc_id == "HD-CLIN-002" and "Scoring table" in c.section)
print("\nchunk id:", example.chunk_id)
print("citation:", example.citation)
print("\n--- what gets embedded ---")
print(example.embedding_text[:600])

The **embedded text starts with the title and section**. A chunk that says only *"call within 30 minutes"* still carries *"Escalation Protocol v2.0 > 4. Escalation by risk band"*, which helps search match it to the right questions. This is sometimes called *contextual chunk headers*.

## 5. Embeddings in five minutes

An **embedding** turns text into a list of numbers (here 384) such that texts with similar **meaning** end up close together. Closeness is measured with **cosine similarity**: 1 = same direction, 0 = unrelated.

We use a small open model, **BAAI/bge-small-en-v1.5**, run locally with `fastembed` (ONNX, no PyTorch, no API key). On Databricks, Vector Search computes embeddings with a hosted model instead.

In [ ]:
embedder = retrieval.FastEmbedder(cache_dir=ROOT / "data" / "models")
hashing = retrieval.HashingEmbedder()

sentences = [
    "SpO2 is low",
    "oxygen saturation dropped",
    "the device battery is flat",
    "the monitor ran out of charge",
    "call the rapid response team",
]


def similarity_matrix(emb):
    v = emb.embed_documents(sentences)
    return pd.DataFrame(v @ v.T, index=sentences, columns=sentences)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
pairs = [("semantic (bge-small)", embedder), ("word-count hashing", hashing)]
for ax, (title, emb) in zip(axes, pairs, strict=True):
    m = similarity_matrix(emb)
    ax.imshow(m, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(sentences)), sentences, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(sentences)), sentences, fontsize=8)
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            ax.text(j, i, f"{m.iat[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
plt.tight_layout()
plt.show()

The semantic model sees that *"SpO2 is low"* and *"oxygen saturation dropped"* mean nearly the same thing, and so do the two battery sentences, **without sharing a word**. The word-count embedder only matches shared words. That's the power of semantic search. Note also that unrelated sentences still score well above 0 with the semantic model (often 0.4-0.6): similarity is **relative**, which matters in section 10.

## 6. A FAISS index and the first searches

**FAISS** stores the chunk vectors and finds the nearest ones to a query vector. With 70 chunks a plain matrix product would do; FAISS keeps the same code working for millions of vectors. We use an exact inner-product index (`IndexFlatIP`); with normalised vectors, inner product = cosine similarity.

In [ ]:
retriever = retrieval.Retriever(chunks, embedder)


def show(results):
    return pd.DataFrame(
        [
            {"rank": r.rank, "similarity": round(r.similarity, 3), "citation": r.chunk.citation}
            for r in results
        ]
    )


show(retriever.search("What should a nurse do when a patient's risk turns High?", 3, "vector"))

In [ ]:
show(retriever.search("Why are oxygen readings too low at Northshore?", k=3, mode="vector"))

The second query never says "SpO2", "field safety notice" or "Site B", yet it finds the notice and the runbook scenario. Semantic search bridges vocabulary.

Save the index: the backend's **demo mode** (Plan.md §13) serves exactly these files after the Databricks credits end.

In [ ]:
index_dir = ROOT / "data" / "rag_local"
retriever.save(index_dir)
print(sorted(p.name for p in index_dir.iterdir()))
reloaded = retrieval.Retriever.load(index_dir, embedder)
print("reloaded:", len(reloaded.chunks), "chunks")

## 7. Keyword and hybrid search

Semantic search has a blind spot: **exact identifiers**. An extension number, a notice ID or a column name has little "meaning" to embed. Classic keyword ranking, **BM25**, is good at exactly that: it rewards rare words that appear in the chunk.

**Hybrid search** runs both and merges the rankings with **reciprocal rank fusion** (RRF): each chunk scores `1/(60 + rank)` in each ranking, summed. It uses ranks only, so cosine and BM25 scores never need to be put on the same scale. (Databricks Vector Search has a hybrid mode too.)

In [ ]:
queries = [
    "bleep 5520",
    "extension 4444",
    "FSN-2026-11",
    "hours_since_obs",
    "the monitor keeps going offline",
    "patients getting worse faster than the model expects",
]
rows = []
for q in queries:
    row = {"query": q}
    for mode in ["vector", "bm25", "hybrid"]:
        top = retriever.search(q, k=1, mode=mode)[0].chunk
        row[mode] = f"{top.doc_id} > {top.section.split(' > ')[-1][:38]}"
    rows.append(row)
pd.DataFrame(rows)

**Reading it:**
- **BM25** nails the bare numbers (`5520`, `4444`) where vector search drifts to unrelated sections. But it fails on paraphrases: "going offline" lands on Wi-Fi release notes, and "getting worse faster than expected" shares no useful words with the protocol change.
- **Vector search** handles the paraphrases ("going offline" finds battery outages and dropped messages; "getting worse faster" finds the protocol change) but not the bare numbers.
- **Hybrid** gets most of them, but not all: for `extension 4444` it picks the protocol section that says *"dial the site RRT extension"*, which both rankings like, over the contacts table that only BM25 liked. When rankings disagree, a chunk ranked fairly well by both can beat one ranked first by only one.

One subtle bug worth knowing (fixed in `reciprocal_rank_fusion`): when BM25 matches **no** words in most chunks, they all tie at score 0 and their tie order is arbitrary. If those arbitrary ranks earn RRF credit, junk gets promoted. A ranking should only vote for items it actually matched.

Which mode is best? Don't guess. Measure.

## 8. Measuring retrieval

The eval set (`retrieval.yaml`) holds ~40 realistic questions, each with the document (and section) that answers it, plus a few the corpus **can't** answer. For each question we record the **rank** of the first correct chunk:
- **hit@1**: correct chunk ranked first;
- **hit@4**: correct chunk anywhere in the top 4 (what we'd pass to the LLM);
- **MRR** (mean reciprocal rank): 1 for rank 1, 0.5 for rank 2, ..., 0 if missed.

This set seeds the Phase 4 agent evaluation, which runs as a CI gate.

In [ ]:
questions = retrieval.load_questions(EVALSET)
results_by_mode = {}
for name, r, mode in [
    ("hashing, vector", retrieval.Retriever(chunks, hashing), "vector"),
    ("BM25", retriever, "bm25"),
    ("bge-small, vector", retriever, "vector"),
    ("bge-small, hybrid", retriever, "hybrid"),
]:
    rows = retrieval.evaluate_retrieval(r, questions, k=4, mode=mode)
    results_by_mode[name] = rows
    results_by_mode[name + " summary"] = retrieval.retrieval_summary(rows, k=4)
pd.DataFrame({k: v for k, v in results_by_mode.items() if k.endswith("summary")}).T.round(3)

In [ ]:
misses = pd.DataFrame(results_by_mode["bge-small, hybrid"])
misses = misses[misses["answerable"] & (misses["rank"].isna() | (misses["rank"] > 1))]
misses[["question", "rank", "top_citation"]]

**Reading these:**
- Word-count hashing is clearly worst: no meaning, and none of BM25's word weighting.
- BM25 is surprisingly strong on a small, consistent corpus written in the same words people ask with.
- The semantic and hybrid set-ups find the right chunk in the top 4 for ~95% of questions. The differences between them are one or two questions, **well within noise** for a 40-question set.
- Many "misses" at rank 2+ are near-misses: a related chunk ranked first (e.g. the runbook's triage checklist above the sensor-bias scenario). With top-4 retrieval the LLM still sees the right passage.

We keep **hybrid** as the default: on this set it's as good as vector-only, and it protects against exact-identifier questions, which real staff ask. If a later eval says otherwise, the setting changes. That's the point of having the eval.

## 9. Document versions

"What NEWS2 score triggers a rapid response call?" has **two** answers: 7 under protocol v1.0 and 5 under v2.0. Searching everything would put both in front of the LLM and invite a muddled answer.

Every chunk carries `status` and `effective_date`, so retrieval can filter:
- by default, only **current** documents;
- with `as_of=<date>`, the versions **in force on that date**, e.g. to explain an escalation decision made last week.

In [ ]:
q = "What NEWS2 score triggers a call to the rapid response team?"
for label, as_of in [("current (default)", None), ("as of 2026-11-05", date(2026, 11, 5))]:
    hits = retriever.search(q, k=3, as_of=as_of)
    print(f"{label}: top hits")
    for r in hits:
        print("   ", r.chunk.citation)
    table = next(r.chunk for r in hits if "| 5-6" in r.chunk.text)
    row = next(line for line in table.text.splitlines() if line.startswith("| 5-6"))
    print("    NEWS2 5-6 row:", row)
    print()

Same question, different correct answer depending on the date. Versioned retrieval also helps the drift story: the concept-drift alert on 2026-11-09 can be explained by retrieving "what changed" from the new protocol.

## 10. Knowing when not to answer

Search always returns *something*, even for "who won the football game?". The assistant must recognise when the documents don't cover a question. A first defence is a **similarity floor**: if the best match is too weak, decline.

In [ ]:
vector_rows = pd.DataFrame(results_by_mode["bge-small, vector"])
fig, ax = plt.subplots(figsize=(8, 2.8))
for y, (answerable, part) in enumerate(vector_rows.groupby("answerable")):
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(part))
    label = "answerable" if answerable else "not in the documents"
    ax.scatter(part["top_similarity"], y + jitter, label=label, alpha=0.7)
ax.set_yticks([0, 1], ["not answerable", "answerable"])
ax.set_xlabel("best cosine similarity")
plt.tight_layout()
plt.show()
lowest = vector_rows.sort_values("top_similarity").head(8)
lowest[["question", "answerable", "top_similarity"]].round(3)

The off-topic questions (football, passwords) sit clearly lower. But plausible hospital questions the documents don't cover score as high as genuine ones: a **parking policy** (0.66) and an **amoxicillin dose** (0.63) both beat "What is FSN-2026-11?" (0.58). The documents are about hospitals too. **No similarity floor separates them cleanly.**

The amoxicillin question is the dangerous kind: retrieval will happily return the pneumonia-outbreak runbook section, and an LLM without strict rules might "help" with a dose. That's why the prompt forbids clinical advice beyond the protocols.

So declining needs two layers:
1. a **low floor** (around 0.55 here) that cheaply rejects clearly off-topic questions before calling the LLM;
2. the **prompt rule** "answer only from the passages; if they don't contain the answer, say so", checked by the Phase 4 eval (groundedness, and refusal on the unanswerable questions).

In [ ]:
FLOOR = 0.55
for q in [
    "Who won the football game last night?",
    "What is the parking policy for night-shift staff?",
    "What is the SpO2 target for COPD patients?",
]:
    res = retriever.search(q, k=4, mode="vector")
    passed = retrieval.answerable(res, FLOOR)
    verdict = "search passed; LLM decides" if passed else "declined before the LLM"
    print(f"{res[0].similarity:.2f}  {verdict:28s}  {q}")

## 11. The prompt the LLM sees

Retrieval's output becomes the LLM's input: rules, numbered passages with citations, then the question. Phase 4 sends this to a Foundation Model API endpoint (or, in demo mode, to the configured LLM); here we just look at it.

In [ ]:
question = "A patient's risk band just turned High. What should I do, and how fast?"
results = retriever.search(question, k=3)
messages = retrieval.build_prompt(question, results)
print(messages[0]["content"])
print("=" * 80)
print(messages[1]["content"][:2500])

A good answer would say: *outreach nurse review within 30 minutes, repeat observations and NEWS2, call the RRT if NEWS2 ≥ 5 [1]*, with the citation pointing to protocol v2.0, section 4. Because each passage carries its document, version and section, the app can turn `[1]` into a link.

**How this maps to Databricks in Phase 4:**

| Here (local / demo mode) | Databricks (live mode) |
|---|---|
| `docs/corpus/*.md` | the same files in a UC Volume |
| `corpus.chunk_corpus` | the same function, writing a Delta table of chunks |
| `FastEmbedder` (bge-small) | a hosted embedding endpoint |
| FAISS `IndexFlatIP` | a Vector Search **Delta Sync** index on the chunk table |
| `Retriever.search(mode="hybrid")` | Vector Search query with hybrid mode |
| `retrieval.yaml` + `evaluate_retrieval` | the MLflow GenAI eval set and scorers |

## 12. Exercise: how big should a chunk be?

Chunk size is a trade-off: small chunks are precise but lose context; big chunks keep context but blur topics (and cost more prompt tokens).

1. rebuild the chunks with `corpus.chunk_corpus(docs, max_words=...)` for 80, 220 (default) and 500
2. for each, build a `retrieval.Retriever` with the **same** `embedder` and compute `retrieval_summary` on the eval set (`mode="hybrid"`)
3. also record the number of chunks and the average words per chunk. Which size would you choose, and why?

In [ ]:
# Try it yourself
chunk_size_results = None

In [ ]:
# Answer (run after trying)
rows = {}
for size in [80, 220, 500]:
    cs = corpus.chunk_corpus(docs, max_words=size)
    r = retrieval.Retriever(cs, embedder)
    evaluated = retrieval.evaluate_retrieval(r, questions, k=4, mode="hybrid")
    avg_words = np.mean([len(c.text.split()) for c in cs])
    rows[size] = {"chunks": len(cs), "avg words": avg_words} | retrieval.retrieval_summary(
        evaluated
    )
chunk_size_results = pd.DataFrame(rows).T.round(3)
chunk_size_results

Our documents are already organised into short sections, so the limit rarely bites: 84, 70 and 69 chunks with almost the same average size, and identical scores. Structure-aware chunking makes retrieval much less sensitive to this knob. On long, unstructured documents (scanned PDFs, emails) it matters far more, both for retrieval quality and for prompt cost (4 chunks of 500 words is a much longer prompt than 4 of 80).

**Stretch:** write a new document in `docs/corpus/` (e.g. a *Visiting Hours Policy*), add two questions about it to a copy of the eval set, rebuild, and check it's found. Then ask the parking question again.

## 13. Pitfalls and next steps

**Pitfall: RAG over table rows.** Numbers belong in SQL. Use RAG for text.

**Pitfall: chunking that ignores structure.** Half a table is worse than no table.

**Pitfall: judging retrieval by a few hand-picked queries.** Build an eval set early and re-run it after every change (chunking, model, mode).

**Pitfall: mixing document versions.** Filter by status or effective date, and cite the version.

**Pitfall: trusting a similarity threshold to stop hallucination.** It only catches the obviously off-topic. The prompt and the evaluation must do the rest.

**Next:** this completes the local notebooks for Phase 0. The next Phase 0 items are the backend and web app in **demo mode** (DuckDB over a gold snapshot, and this FAISS index), then the Phase 1 Databricks notebooks.